# Telecom Customer Churn Prediction & Model Evaluation
### Scikit-Learn Pipelines, Logistic Regression, Random Forest & Feature Importances

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
%matplotlib inline

## 1. Load Processed Data & Define Feature Sets

In [ ]:
df = pd.read_csv('../data/processed/cleaned_churn.csv')
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop(columns=['customerID', 'Churn', 'Churn_Numeric', 'SeniorCitizen_Label', 'Tenure_Group', 'Avg_Monthly_Paid'])

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training shape: {X_train.shape}, Testing shape: {X_test.shape}')

## 2. Train Models (Logistic Regression & Random Forest)

In [ ]:
lr_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_pipe.fit(X_train, y_train)

rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train)
print('Both models trained successfully.')

## 3. Model Evaluation & Comparison

In [ ]:
results = []
for name, pipe in [('Logistic Regression', lr_pipe), ('Random Forest', rf_pipe)]:
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': f'{accuracy_score(y_test, y_pred)*100:.2f}%',
        'Precision': f'{precision_score(y_test, y_pred)*100:.2f}%',
        'Recall': f'{recall_score(y_test, y_pred)*100:.2f}%',
        'F1-Score': f'{f1_score(y_test, y_pred):.4f}',
        'ROC-AUC': f'{roc_auc_score(y_test, y_prob):.4f}'
    })
pd.DataFrame(results)